# 06 - Gold et couche de service

Dans cette étape je pars de la table silver et je calcule 5 petites tables gold, une par graphique du dashboard :
- gold_mensuel : nombre de séismes et magnitude max par mois (M >= 4.5)
- gold_magnitudes : nombre de séismes par classe de magnitude (toutes les magnitudes), pour Gutenberg-Richter et la valeur b
- gold_profondeur : nombre de séismes par classe de profondeur (M >= 4.5)
- gold_carte : la liste des séismes pour la carte (M >= 4.5)
- gold_qualite : le bilan qualité (lignes reçues, hors périmètre, rejetées)

Toutes les tables gardent la colonne annee pour que le filtre par années du dashboard marche partout.

Ensuite j'exporte ces tables en Parquet dans data/serving : c'est ce que le dashboard lira (il ne lit pas HDFS directement).

## 1. J'ouvre la SparkSession et je définis les chemins

In [1]:
from pathlib import Path
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("06_gold_serving").getOrCreate()

CHEMIN_BRONZE = "/lakehouse/bronze/seismes"
CHEMIN_SILVER = "/lakehouse/silver/seismes"
CHEMIN_REJETS = "/lakehouse/silver_rejets/seismes"
CHEMIN_GOLD = "/lakehouse/gold"                    # dans HDFS
DOSSIER_SERVING = Path("/work/data/serving")       # = data/serving sur mon PC

MC = 4.5   # magnitude de complétude : en dessous, les petits séismes sont mal détectés hors des États-Unis

print("Application :", spark.sparkContext.appName)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/12 16:03:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Application : 06_gold_serving


## 2. Je lis silver et je garde seulement les colonnes utiles
Je mets en cache car toutes les tables gold partent de ces données.
forts = les séismes de magnitude >= 4.5, utilisés par la plupart des tables.

In [2]:
seismes = (
    spark.read.format("delta").load(CHEMIN_SILVER)
    .select("id", "date", "annee", "mois", "latitude", "longitude",
            "profondeur", "classe_profondeur", "magnitude", "lieu")
    .cache()
)
forts = seismes.filter(F.col("magnitude") >= MC)

nb_seismes = seismes.count()
nb_forts = forts.count()
print("Séismes dans silver          :", nb_seismes)
print("Séismes de magnitude >= 4.5  :", nb_forts)

26/11/12 16:03:56 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Séismes dans silver          : 1566506
Séismes de magnitude >= 4.5  : 74362


## 3. Je crée une petite fonction pour écrire une table gold
Chaque table gold est écrite en Delta dans HDFS. Elles sont petites, donc coalesce(1) pour avoir un seul fichier.

In [3]:
def ecrire_gold(df, nom):
    chemin = f"{CHEMIN_GOLD}/{nom}"
    df.coalesce(1).write.format("delta").mode("overwrite").save(chemin)
    nb = spark.read.format("delta").load(chemin).count()
    print(f"{nom} écrite : {nb} lignes")
    return nb

## 4. gold_mensuel : nombre de séismes et magnitude max par mois (M >= 4.5)
Pour le graphique de l'évolution mensuelle et pour les indicateurs du dashboard.

In [4]:
gold_mensuel = (
    forts.groupBy("annee", "mois")
    .agg(F.count("*").alias("nb_seismes"),
         F.max("magnitude").alias("mag_max"))
    .orderBy("annee", "mois")
)
ecrire_gold(gold_mensuel, "gold_mensuel")

# vérification : en additionnant tous les mois je dois retrouver le nombre de séismes >= 4.5
total = gold_mensuel.agg(F.sum("nb_seismes")).first()[0]
print("Somme des mois :", total, "| attendu :", nb_forts, "|", total == nb_forts)

print("Les 5 mois avec le plus de séismes :")
gold_mensuel.orderBy(F.desc("nb_seismes")).show(5)

gold_mensuel écrite : 120 lignes


Somme des mois : 74362 | attendu : 74362 | True
Les 5 mois avec le plus de séismes :


+-----+----+----------+-------+
|annee|mois|nb_seismes|mag_max|
+-----+----+----------+-------+
| 2025|   7|      1321|    8.8|
| 2021|   8|      1307|    8.1|
| 2021|   3|      1262|    8.1|
| 2023|  12|      1117|    7.6|
| 2025|   8|       983|    7.5|
+-----+----+----------+-------+
only showing top 5 rows



## 5. gold_magnitudes : nombre de séismes par classe de magnitude (toutes les magnitudes)
J'arrondis la magnitude à 0.1 près : chaque arrondi est une classe (4.5, 4.6, 4.7...).
Ici je garde toutes les magnitudes pour voir sur la courbe de Gutenberg-Richter où les petits séismes commencent à manquer.

In [5]:
gold_magnitudes = (
    seismes.withColumn("mag_classe", F.round("magnitude", 1))
    .groupBy("annee", "mag_classe")
    .agg(F.count("*").alias("nb_seismes"))
    .orderBy("annee", "mag_classe")
)
ecrire_gold(gold_magnitudes, "gold_magnitudes")

total = gold_magnitudes.agg(F.sum("nb_seismes")).first()[0]
print("Somme des classes :", total, "| attendu :", nb_seismes, "|", total == nb_seismes)

gold_magnitudes écrite : 951 lignes


Somme des classes : 1566506 | attendu : 1566506 | True


## 6. gold_profondeur : nombre de séismes par classe de profondeur (M >= 4.5)
superficiel (moins de 70 km), intermediaire (70 à 300 km), profond (plus de 300 km)

In [6]:
gold_profondeur = (
    forts.groupBy("annee", "classe_profondeur")
    .agg(F.count("*").alias("nb_seismes"))
    .orderBy("annee", "classe_profondeur")
)
ecrire_gold(gold_profondeur, "gold_profondeur")

total = gold_profondeur.agg(F.sum("nb_seismes")).first()[0]
print("Somme des classes :", total, "| attendu :", nb_forts, "|", total == nb_forts)

print("Répartition sur les 10 ans :")
(
    gold_profondeur.groupBy("classe_profondeur")
    .agg(F.sum("nb_seismes").alias("nb_seismes"))
    .withColumn("pourcentage", F.round(F.col("nb_seismes") * 100 / nb_forts, 1))
    .orderBy(F.desc("nb_seismes"))
    .show()
)

gold_profondeur écrite : 30 lignes
Somme des classes : 74362 | attendu : 74362 | True
Répartition sur les 10 ans :
+-----------------+----------+-----------+
|classe_profondeur|nb_seismes|pourcentage|
+-----------------+----------+-----------+
|      superficiel|     58719|       79.0|
|    intermediaire|     12199|       16.4|
|          profond|      3444|        4.6|
+-----------------+----------+-----------+



## 7. gold_carte : un point par séisme pour la carte (M >= 4.5)
Ici je ne fais pas d'agrégation, je garde chaque séisme avec sa position. Avec le filtre M >= 4.5 ça reste raisonnable pour une carte.

In [7]:
gold_carte = forts.select("id", "date", "annee", "latitude", "longitude",
                          "profondeur", "magnitude", "lieu")
nb_carte = ecrire_gold(gold_carte, "gold_carte")
print("Attendu :", nb_forts, "|", nb_carte == nb_forts)

print("Les 5 plus forts séismes de la période :")
gold_carte.orderBy(F.desc("magnitude")).select("date", "magnitude", "profondeur", "lieu").show(5, truncate=False)

gold_carte écrite : 74362 lignes
Attendu : 74362 | True
Les 5 plus forts séismes de la période :


+-----------------------+---------+----------+---------------------------------------------+
|date                   |magnitude|profondeur|lieu                                         |
+-----------------------+---------+----------+---------------------------------------------+
|2025-07-29 23:24:52.483|8.8      |35.0      |2025 Kamchatka Peninsula, Russia Earthquake  |
|2018-08-19 00:19:40.67 |8.2      |600.0     |2018 Fiji Earthquake                         |
|2017-09-08 04:49:19.18 |8.2      |47.39     |2017 Tehuantepec, Mexico Earthquake          |
|2021-07-29 06:15:49.188|8.2      |35.0      |2021 Chignik, Alaska Earthquake              |
|2021-03-04 19:28:33.178|8.1      |28.93     |2021 Kermadec Islands, New Zealand Earthquake|
+-----------------------+---------+----------+---------------------------------------------+
only showing top 5 rows



## 8. gold_qualite : le bilan qualité
Je reprends les comptes de bronze, des rejets et de silver pour avoir une table : etape, motif, nb_lignes.
Les doublons, je les retrouve par différence (reçues - hors périmètre - rejetées - gardées).

In [8]:
bronze = spark.read.format("delta").load(CHEMIN_BRONZE)
rejets = spark.read.format("delta").load(CHEMIN_REJETS)

nb_recues = bronze.count()
lignes = [("recues", "toutes", nb_recues)]

# hors périmètre : un motif par type d'événement (explosion, quarry blast...)
hors = (
    bronze.withColumn("type_propre", F.lower(F.trim(F.col("type"))))
    .filter((F.col("type_propre") != "earthquake") | F.col("type_propre").isNull())
    .groupBy("type_propre").count()
    .collect()
)
for ligne in hors:
    lignes.append(("hors_perimetre", ligne["type_propre"], ligne["count"]))

# rejetées : un motif par règle de qualité
for ligne in rejets.groupBy("motif").count().collect():
    lignes.append(("rejetees", ligne["motif"], ligne["count"]))

lignes.append(("gardees", "silver", nb_seismes))

somme = sum(nb for etape, motif, nb in lignes if etape != "recues")
lignes.append(("doublons", "meme_id", nb_recues - somme))

gold_qualite = spark.createDataFrame(lignes, ["etape", "motif", "nb_lignes"])
ecrire_gold(gold_qualite, "gold_qualite")

gold_qualite.groupBy("etape").agg(F.sum("nb_lignes").alias("nb_lignes")).orderBy(F.desc("nb_lignes")).show()

gold_qualite écrite : 23 lignes


+--------------+---------+
|         etape|nb_lignes|
+--------------+---------+
|        recues|  1604291|
|       gardees|  1566506|
|hors_perimetre|    37170|
|      rejetees|      615|
|      doublons|        0|
+--------------+---------+



## 9. Aperçu de la valeur b (loi de Gutenberg-Richter)
Le dashboard la recalculera selon les années choisies, mais je la calcule ici une fois sur les 10 ans pour vérifier.
Estimateur d'Aki : b = log10(e) / (M moyenne - (Mc - 0.05)), avec seulement les classes >= Mc = 4.5.
Pour la Terre entière on s'attend à une valeur proche de 1.

In [9]:
import math

mag = gold_magnitudes.toPandas()                         # petite table, je peux la passer en pandas
mag = mag[mag["mag_classe"] >= MC]

m_moyenne = (mag["mag_classe"] * mag["nb_seismes"]).sum() / mag["nb_seismes"].sum()
b = math.log10(math.e) / (m_moyenne - (MC - 0.05))

print("Séismes utilisés (classes >= 4.5) :", mag["nb_seismes"].sum())
print(f"Magnitude moyenne                 : {m_moyenne:.3f}")
print(f"Valeur b (Aki)                    : {b:.2f}")

Séismes utilisés (classes >= 4.5) : 74392
Magnitude moyenne                 : 4.804
Valeur b (Aki)                    : 1.23


## 10. J'exporte les tables gold en Parquet dans data/serving (couche de service)
Les tables sont petites, donc le driver (ce notebook) les récupère avec toPandas() et écrit lui-même les fichiers.
C'est pour ça que seul le conteneur jupyter a besoin du dossier data/serving.

In [10]:
DOSSIER_SERVING.mkdir(parents=True, exist_ok=True)

for nom in ["gold_mensuel", "gold_magnitudes", "gold_profondeur", "gold_carte", "gold_qualite"]:
    df = spark.read.format("delta").load(f"{CHEMIN_GOLD}/{nom}").toPandas()
    df.to_parquet(DOSSIER_SERVING / f"{nom}.parquet", index=False)
    print(f"{nom}.parquet : {len(df)} lignes")

gold_mensuel.parquet : 120 lignes


gold_magnitudes.parquet : 951 lignes


gold_profondeur.parquet : 30 lignes


gold_carte.parquet : 74362 lignes


gold_qualite.parquet : 23 lignes


## 11. Je vérifie les fichiers de la couche de service
Je les relis avec pandas seulement (sans Spark), comme le fera le dashboard.

In [11]:
for fichier in sorted(DOSSIER_SERVING.glob("*.parquet")):
    df = pd.read_parquet(fichier)
    print(f"{fichier.name:<24} {len(df):>7} lignes  {fichier.stat().st_size / 1e3:>7.0f} Ko  colonnes : {list(df.columns)}")

gold_carte.parquet         74362 lignes     3653 Ko  colonnes : ['id', 'date', 'annee', 'latitude', 'longitude', 'profondeur', 'magnitude', 'lieu']
gold_magnitudes.parquet      951 lignes        8 Ko  colonnes : ['annee', 'mag_classe', 'nb_seismes']
gold_mensuel.parquet         120 lignes        4 Ko  colonnes : ['annee', 'mois', 'nb_seismes', 'mag_max']
gold_profondeur.parquet       30 lignes        3 Ko  colonnes : ['annee', 'classe_profondeur', 'nb_seismes']
gold_qualite.parquet          23 lignes        3 Ko  colonnes : ['etape', 'motif', 'nb_lignes']


## 12. Je libère le cache et je ferme la SparkSession

In [12]:
seismes.unpersist()
spark.stop()